# Lab – Large Language Models
## Partie 1 : Inférence sans entraînement

### Question 1 : Chargement du dataset

In [1]:
!pip install -q transformers datasets torch scikit-learn pandas

In [3]:
from datasets import load_dataset

data = load_dataset("cornell-movie-review-data/rotten_tomatoes")

README.md:   0%|          | 0.00/7.46k [00:00<?, ?B/s]

train.parquet: reconstructing file:   0%|          |  0.00B /  699kB            

train.parquet: downloading bytes:           |  0.00B            

validation.parquet: reconstructing file:   0%|          |  0.00B / 90.0kB            

validation.parquet: downloading bytes:           |  0.00B            

test.parquet: reconstructing file:   0%|          |  0.00B / 92.2kB            

test.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/8530 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1066 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1066 [00:00<?, ? examples/s]

### Question 2 : Structure du dataset

In [4]:
print(data)

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 8530
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 1066
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 1066
    })
})


### Question 3 : Premier échantillon du split d'entraînement

In [5]:
print(data["train"][0])

{'text': 'the rock is destined to be the 21st century\'s new " conan " and that he\'s going to make a splash even greater than arnold schwarzenegger , jean-claud van damme or steven segal .', 'label': 1}


### Question 4 : Chargement du modèle avec `pipeline`

In [6]:
from transformers import pipeline
import torch

model_path = "cardiffnlp/twitter-roberta-base-sentiment-latest"

pipe = pipeline(
    "sentiment-analysis",
    model=model_path,
    tokenizer=model_path,
    return_all_scores=True,
    device=0 if torch.cuda.is_available() else -1,
)

config.json:   0%|          | 0.00/929 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  501MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  501MB            

model.safetensors: downloading bytes:           |  0.00B            

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

test rapide

In [7]:
print(pipe("I love this movie!"))

[{'label': 'positive', 'score': 0.9847092628479004}]


### Question 5 : Inférence sur le split test


Recharger le pipeline avec tous les scores

In [8]:
pipe = pipeline(
    "sentiment-analysis",
    model=model_path,
    tokenizer=model_path,
    top_k=None,
    device=0 if torch.cuda.is_available() else -1,
)
print(pipe("I love this movie!"))

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[[{'label': 'positive', 'score': 0.9847092628479004}, {'label': 'neutral', 'score': 0.01182967983186245}, {'label': 'negative', 'score': 0.003461067797616124}]]


Inférence

In [10]:
texts = list(data["test"]["text"])
y_true = list(data["test"]["label"])

all_scores = pipe(texts, batch_size=32, truncation=True)

y_pred = []
for scores in all_scores:
    s = {d["label"].lower(): d["score"] for d in scores}
    y_pred.append(1 if s["positive"] > s["negative"] else 0)

print(len(y_pred), y_pred[:10])

1066 [1, 1, 0, 1, 1, 0, 1, 1, 1, 1]


### Question 6 : Rapport de classification

In [11]:
from sklearn.metrics import classification_report

print(classification_report(y_true, y_pred, target_names=["negative", "positive"], digits=4))

              precision    recall  f1-score   support

    negative     0.7589    0.8799    0.8149       533
    positive     0.8571    0.7205    0.7829       533

    accuracy                         0.8002      1066
   macro avg     0.8080    0.8002    0.7989      1066
weighted avg     0.8080    0.8002    0.7989      1066



### Question 7 : Tokens, IDs et embeddings

In [12]:
from transformers import AutoTokenizer, AutoModel
import torch
import pandas as pd

tokenizer = AutoTokenizer.from_pretrained("cardiffnlp/twitter-roberta-base-sentiment-latest")
model = AutoModel.from_pretrained("cardiffnlp/twitter-roberta-base-sentiment-latest")
model.eval()

prompt = "I love this movie!"

inputs = tokenizer(prompt, return_tensors="pt")
input_ids = inputs["input_ids"][0].tolist()

with torch.no_grad():
    outputs = model(**inputs)

token_embeddings = outputs.last_hidden_state[0]

rows = []
for tok_id, embedding in zip(input_ids, token_embeddings):
    rows.append({
        "Token": tokenizer.decode(tok_id),
        "ID": tok_id,
        "Embedding": embedding.numpy(),
    })

df = pd.DataFrame(rows)
print(df)
print("Shape des embeddings :", token_embeddings.shape)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                        | Status     |  | 
---------------------------+------------+--+-
classifier.dense.bias      | UNEXPECTED |  | 
classifier.dense.weight    | UNEXPECTED |  | 
classifier.out_proj.weight | UNEXPECTED |  | 
classifier.out_proj.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


    Token    ID                                          Embedding
0     <s>     0  [-0.057009406, 0.6744933, -0.24589871, -0.4016...
1       I   100  [-0.08691575, 0.25903293, -0.22076868, 0.10727...
2    love   657  [-0.34278533, 0.4889539, 0.054818142, 0.023185...
3    this    42  [-0.18056487, 0.20334247, 0.1408535, -0.064378...
4   movie  1569  [-0.35465932, 0.2917426, 0.101844035, -0.03130...
5       !   328  [-0.11802508, 0.16298923, 0.20535733, 0.113921...
6    </s>     2  [-0.057030655, 0.67452806, -0.2458932, -0.4015...
Shape des embeddings : torch.Size([7, 768])


## Partie 2 : Linear probing

### Question 1 : Gel du modèle (sauf la tête de classification)

Tokenisation

In [13]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding

model_path = "cardiffnlp/twitter-roberta-base-sentiment-latest"
tokenizer = AutoTokenizer.from_pretrained(model_path)

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True)

tokenized = data.map(tokenize, batched=True)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

Map:   0%|          | 0/8530 [00:00<?, ? examples/s]

Map:   0%|          | 0/1066 [00:00<?, ? examples/s]

Map:   0%|          | 0/1066 [00:00<?, ? examples/s]

Modèle et gel

In [14]:
model_lp = AutoModelForSequenceClassification.from_pretrained(
    model_path,
    num_labels=2,
    id2label={0: "negative", 1: "positive"},
    label2id={"negative": 0, "positive": 1},
    ignore_mismatched_sizes=True,
)

for param in model_lp.roberta.parameters():
    param.requires_grad = False

for name, param in model_lp.named_parameters():
    if param.requires_grad:
        print("Entraînable :", name)

trainable = sum(p.numel() for p in model_lp.parameters() if p.requires_grad)
total = sum(p.numel() for p in model_lp.parameters())
print(f"Paramètres entraînables : {trainable:,} / {total:,} ({100 * trainable / total:.2f} %)")

[transformers] You passed `num_labels=2` which is incompatible to the `id2label` map of length `3`.


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |                                                                                       
----------------------------+------------+---------------------------------------------------------------------------------------
roberta.pooler.dense.bias   | UNEXPECTED |                                                                                       
roberta.pooler.dense.weight | UNEXPECTED |                                                                                       
classifier.out_proj.weight  | MISMATCH   | Reinit due to size mismatch - ckpt: torch.Size([3, 768]) vs model:torch.Size([2, 768])
classifier.out_proj.bias    | MISMATCH   | Reinit due to size mismatch - ckpt: torch.Size([3]) vs model:torch.Size([2])          

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect 

Entraînable : classifier.dense.weight
Entraînable : classifier.dense.bias
Entraînable : classifier.out_proj.weight
Entraînable : classifier.out_proj.bias
Paramètres entraînables : 592,130 / 124,647,170 (0.48 %)


### Question 2 : Entraînement de la tête de classification et comparaison avec la Partie 1

Métrique et entraînement

In [15]:
import numpy as np
import torch
from sklearn.metrics import accuracy_score, f1_score, classification_report
from transformers import TrainingArguments, Trainer

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
    }

args_lp = TrainingArguments(
    output_dir="./lp_output",
    learning_rate=1e-3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="no",
    logging_strategy="epoch",
    fp16=torch.cuda.is_available(),
    report_to="none",
    seed=42,
)

trainer_lp = Trainer(
    model=model_lp,
    args=args_lp,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer_lp.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,0.428371,0.368749,0.845216,0.844740
2,0.394819,0.342465,0.850844,0.850734
3,0.385690,0.335729,0.854597,0.854593


TrainOutput(global_step=801, training_loss=0.40295995933732737, metrics={'train_runtime': 43.1989, 'train_samples_per_second': 592.377, 'train_steps_per_second': 18.542, 'total_flos': 688251243054000.0, 'train_loss': 0.40295995933732737, 'epoch': 3.0})

Evaluation sur le test

In [16]:
pred_lp = trainer_lp.predict(tokenized["test"])
y_pred_lp = np.argmax(pred_lp.predictions, axis=-1)
y_true_lp = pred_lp.label_ids

print(classification_report(y_true_lp, y_pred_lp, target_names=["negative", "positive"], digits=4))

              precision    recall  f1-score   support

    negative     0.8223    0.8424    0.8323       533
    positive     0.8385    0.8180    0.8281       533

    accuracy                         0.8302      1066
   macro avg     0.8304    0.8302    0.8302      1066
weighted avg     0.8304    0.8302    0.8302      1066



Tableau comparatif avec la partie 1

In [17]:
import pandas as pd

results = pd.DataFrame({
    "Méthode": ["Partie 1 : inférence sans entraînement", "Partie 2 : linear probing"],
    "Accuracy": [accuracy_score(y_true, y_pred), accuracy_score(y_true_lp, y_pred_lp)],
    "F1 macro": [f1_score(y_true, y_pred, average="macro"), f1_score(y_true_lp, y_pred_lp, average="macro")],
})
print(results.round(4))

                                  Méthode  Accuracy  F1 macro
0  Partie 1 : inférence sans entraînement    0.8002    0.7989
1               Partie 2 : linear probing    0.8302    0.8302


## Partie 3 : Fine-tuning

### Question 1 : Fine-tuning du modèle complet

Modèle et entraînement

In [18]:
model_ft = AutoModelForSequenceClassification.from_pretrained(
    model_path,
    num_labels=2,
    ignore_mismatched_sizes=True,
)

for param in model_ft.parameters():
    param.requires_grad = True

trainable = sum(p.numel() for p in model_ft.parameters() if p.requires_grad)
total = sum(p.numel() for p in model_ft.parameters())
print(f"Paramètres entraînables : {trainable:,} / {total:,} ({100 * trainable / total:.2f} %)")

args_ft = TrainingArguments(
    output_dir="./ft_output",
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="no",
    logging_strategy="epoch",
    fp16=torch.cuda.is_available(),
    report_to="none",
    seed=42,
)

trainer_ft = Trainer(
    model=model_ft,
    args=args_ft,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer_ft.train()

[transformers] You passed `num_labels=2` which is incompatible to the `id2label` map of length `3`.


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |                                                                                       
----------------------------+------------+---------------------------------------------------------------------------------------
roberta.pooler.dense.bias   | UNEXPECTED |                                                                                       
roberta.pooler.dense.weight | UNEXPECTED |                                                                                       
classifier.out_proj.weight  | MISMATCH   | Reinit due to size mismatch - ckpt: torch.Size([3, 768]) vs model:torch.Size([2, 768])
classifier.out_proj.bias    | MISMATCH   | Reinit due to size mismatch - ckpt: torch.Size([3]) vs model:torch.Size([2])          

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect 

Paramètres entraînables : 124,647,170 / 124,647,170 (100.00 %)


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,0.338993,0.273285,0.890244,0.890112
2,0.216626,0.291734,0.896811,0.896793
3,0.144687,0.355729,0.889306,0.889292


TrainOutput(global_step=801, training_loss=0.2334356748507115, metrics={'train_runtime': 98.1488, 'train_samples_per_second': 260.727, 'train_steps_per_second': 8.161, 'total_flos': 688251243054000.0, 'train_loss': 0.2334356748507115, 'epoch': 3.0})

Evaluation sur le test

In [19]:
pred_ft = trainer_ft.predict(tokenized["test"])
y_pred_ft = np.argmax(pred_ft.predictions, axis=-1)
y_true_ft = pred_ft.label_ids

print(classification_report(y_true_ft, y_pred_ft, target_names=["negative", "positive"], digits=4))

              precision    recall  f1-score   support

    negative     0.8755    0.8968    0.8860       533
    positive     0.8942    0.8724    0.8832       533

    accuracy                         0.8846      1066
   macro avg     0.8848    0.8846    0.8846      1066
weighted avg     0.8848    0.8846    0.8846      1066



Tableau comparatif

In [20]:
results = pd.DataFrame({
    "Méthode": [
        "Partie 1 : inférence sans entraînement",
        "Partie 2 : linear probing",
        "Partie 3 : fine-tuning complet",
    ],
    "Accuracy": [
        accuracy_score(y_true, y_pred),
        accuracy_score(y_true_lp, y_pred_lp),
        accuracy_score(y_true_ft, y_pred_ft),
    ],
    "F1 macro": [
        f1_score(y_true, y_pred, average="macro"),
        f1_score(y_true_lp, y_pred_lp, average="macro"),
        f1_score(y_true_ft, y_pred_ft, average="macro"),
    ],
})
print(results.round(4))

                                  Méthode  Accuracy  F1 macro
0  Partie 1 : inférence sans entraînement    0.8002    0.7989
1               Partie 2 : linear probing    0.8302    0.8302
2          Partie 3 : fine-tuning complet    0.8846    0.8846


### Question 2 : Fine-tuning avec une partie de l'encodeur gelée

Modèle et gel partiel

In [21]:
N_FROZEN = 6

model_pf = AutoModelForSequenceClassification.from_pretrained(
    model_path,
    num_labels=2,
    ignore_mismatched_sizes=True,
)

for param in model_pf.roberta.embeddings.parameters():
    param.requires_grad = False

for layer in model_pf.roberta.encoder.layer[:N_FROZEN]:
    for param in layer.parameters():
        param.requires_grad = False

for name, param in model_pf.named_parameters():
    if param.requires_grad and "encoder.layer" in name:
        print("Entraînable :", name.split(".weight")[0].split(".bias")[0])
        break

trainable = sum(p.numel() for p in model_pf.parameters() if p.requires_grad)
total = sum(p.numel() for p in model_pf.parameters())
print(f"Paramètres entraînables : {trainable:,} / {total:,} ({100 * trainable / total:.2f} %)")

[transformers] You passed `num_labels=2` which is incompatible to the `id2label` map of length `3`.


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |                                                                                       
----------------------------+------------+---------------------------------------------------------------------------------------
roberta.pooler.dense.bias   | UNEXPECTED |                                                                                       
roberta.pooler.dense.weight | UNEXPECTED |                                                                                       
classifier.out_proj.weight  | MISMATCH   | Reinit due to size mismatch - ckpt: torch.Size([3, 768]) vs model:torch.Size([2, 768])
classifier.out_proj.bias    | MISMATCH   | Reinit due to size mismatch - ckpt: torch.Size([3]) vs model:torch.Size([2])          

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect 

Entraînable : roberta.encoder.layer.6.attention.self.query
Paramètres entraînables : 43,119,362 / 124,647,170 (34.59 %)


Entraînement et évaluation

In [22]:
args_pf = TrainingArguments(
    output_dir="./pf_output",
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="no",
    logging_strategy="epoch",
    fp16=torch.cuda.is_available(),
    report_to="none",
    seed=42,
)

trainer_pf = Trainer(
    model=model_pf,
    args=args_pf,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer_pf.train()

pred_pf = trainer_pf.predict(tokenized["test"])
y_pred_pf = np.argmax(pred_pf.predictions, axis=-1)
y_true_pf = pred_pf.label_ids

print(classification_report(y_true_pf, y_pred_pf, target_names=["negative", "positive"], digits=4))

Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,0.359164,0.291627,0.873358,0.873045
2,0.287299,0.276045,0.883677,0.883674
3,0.248818,0.295514,0.883677,0.883667


              precision    recall  f1-score   support

    negative     0.8707    0.8593    0.8650       533
    positive     0.8611    0.8724    0.8667       533

    accuracy                         0.8659      1066
   macro avg     0.8659    0.8659    0.8658      1066
weighted avg     0.8659    0.8659    0.8658      1066



Tableau comparatif

In [23]:
results = pd.DataFrame({
    "Méthode": [
        "Partie 1 : inférence sans entraînement",
        "Partie 2 : linear probing",
        "Partie 3.1 : fine-tuning complet",
        f"Partie 3.2 : fine-tuning, {N_FROZEN} couches gelées",
    ],
    "Accuracy": [
        accuracy_score(y_true, y_pred),
        accuracy_score(y_true_lp, y_pred_lp),
        accuracy_score(y_true_ft, y_pred_ft),
        accuracy_score(y_true_pf, y_pred_pf),
    ],
    "F1 macro": [
        f1_score(y_true, y_pred, average="macro"),
        f1_score(y_true_lp, y_pred_lp, average="macro"),
        f1_score(y_true_ft, y_pred_ft, average="macro"),
        f1_score(y_true_pf, y_pred_pf, average="macro"),
    ],
    "Params entraînables": [0, 592130, 124647170, trainable],
})
print(results.round(4))

                                      Méthode  Accuracy  F1 macro  \
0      Partie 1 : inférence sans entraînement    0.8002    0.7989   
1                   Partie 2 : linear probing    0.8302    0.8302   
2            Partie 3.1 : fine-tuning complet    0.8846    0.8846   
3  Partie 3.2 : fine-tuning, 6 couches gelées    0.8659    0.8658   

   Params entraînables  
0                    0  
1               592130  
2            124647170  
3             43119362  
